In [1]:
import os
os.environ["PYTHONUTF8"] = "1"

In [2]:
%pip install -q transformers==4.57.1 datasets accelerate bitsandbytes peft evaluate sacrebleu sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


In [3]:
import gc
import json
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

In [4]:
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


In [5]:
dataset = load_dataset("opus_books", "en-es")
dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

def rename_columns(example):
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["es"],
    }

dataset = dataset.map(rename_columns, remove_columns=dataset["train"].column_names)

# Debug-safe subset first
dataset["train"] = dataset["train"].select(range(2000))
dataset["test"] = dataset["test"].select(range(200))

print(dataset)
print("Train size:", len(dataset["train"]))
print("Test size:", len(dataset["test"]))
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['src', 'tgt'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['src', 'tgt'],
        num_rows: 200
    })
})
Train size: 2000
Test size: 200
{'src': '"That which has happened me in meeting you, mighty prince," replied Don Quixote, "cannot be unfortunate, even if my fall had not stopped short of the depths of the bottomless pit, for the glory of having seen you would have lifted me up and delivered me from it.', 'tgt': '-El que yo he tenido en veros, valeroso príncipe -respondió don Quijote-, es imposible ser malo, aunque mi caída no parara hasta el profundo de los abismos, pues de allí me levantara y me sacara la gloria de haberos visto.'}


In [6]:
prefix = "translate English to Spanish: "

def build_input(src):
    return prefix + src

print(build_input("Hello, how are you?"))

translate English to Spanish: Hello, how are you?


In [7]:
model_id = "google/flan-t5-small"

baseline_tokenizer = AutoTokenizer.from_pretrained(model_id)

baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

baseline_model.eval()

print("Baseline loaded:", model_id)

`torch_dtype` is deprecated! Use `dtype` instead!


Baseline loaded: google/flan-t5-small


In [8]:
def generate_translation(model, tokenizer, src, max_new_tokens=80):
    text = build_input(src)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [9]:
sample_text = dataset["test"][0]["src"]
reference_text = dataset["test"][0]["tgt"]

baseline_prediction = generate_translation(
    baseline_model,
    baseline_tokenizer,
    sample_text,
)

print("SOURCE:")
print(sample_text)

print("\nREFERENCE:")
print(reference_text)

print("\nBASELINE OUTPUT:")
print(baseline_prediction)

SOURCE:
Kitty, on the contrary, was more active than usual and even more animated.

REFERENCE:
Kitty, al contrario, estaba más activa a incluso más animada que nunca.

BASELINE OUTPUT:
Kitty, en el contrario, fue más activo que el habitado y incluso más animated.


In [10]:
test_subset = dataset["test"].select(range(20))

sources = []
references = []
baseline_predictions = []

for example in test_subset:
    src = example["src"]
    tgt = example["tgt"]

    pred = generate_translation(
        baseline_model,
        baseline_tokenizer,
        src,
    )

    sources.append(src)
    references.append(tgt)
    baseline_predictions.append(pred)

print("Done generating baseline predictions.")
print("Example baseline:", baseline_predictions[0])

Done generating baseline predictions.
Example baseline: Kitty, en el contrario, fue más activo que el habitado y incluso más animated.


In [11]:
bleu = evaluate.load("sacrebleu")

baseline_bleu = bleu.compute(
    predictions=baseline_predictions,
    references=[[ref] for ref in references],
)

print("Baseline BLEU:")
print(baseline_bleu)

Baseline BLEU:
{'score': 6.639058556402083, 'counts': [140, 46, 15, 5], 'totals': [399, 379, 359, 339], 'precisions': [35.08771929824562, 12.137203166226913, 4.178272980501393, 1.4749262536873156], 'bp': 0.9275691148439936, 'sys_len': 399, 'ref_len': 429}


In [12]:
del baseline_model
gc.collect()
torch.cuda.empty_cache()

print("Baseline cleared.")

Baseline cleared.


In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("4-bit model loaded:", model_id)

4-bit model loaded: google/flan-t5-small


In [14]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


In [15]:
max_input_length = 128
max_target_length = 128

def preprocess_batch(examples):
    inputs = [build_input(src) for src in examples["src"]]
    targets = examples["tgt"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print(tokenized)
print(tokenized["train"][0].keys())

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [16]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("data_collator created")

data_collator created


In [17]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan_t5_qlora_translation",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=3,
    logging_steps=20,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    max_grad_norm=0.3,
)

print("training_args created")

training_args created


In [18]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Trainer initialized.")

Trainer initialized.


C:\Users\tyier\AppData\Local\Temp\ipykernel_12740\3288422033.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [19]:
batch = data_collator([tokenized["train"][0], tokenized["train"][1]])

print(batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)
print("labels unique:", torch.unique(batch["labels"]))
print("non-masked labels:", (batch["labels"] != -100).sum())

KeysView({'input_ids': tensor([[13959,  1566,    12,  5093,    10,    96, 11880,    84,    65,  2817,
           140,    16,  1338,    25,     6,     3, 21553, 22277,   976, 18606,
          1008,  6590,   226,    32,    17,    15,     6,    96,  1608,  2264,
            36, 20343,     6,   237,     3,    99,    82,  1590,   141,    59,
          4910,   710,    13,     8,  4963,     7,    13,     8,  2007,   924,
          7688,     6,    21,     8, 12582,    13,   578,   894,    25,   133,
            43, 19464,   140,    95,    11,  3566,   140,    45,    34,     5,
             1,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [13959,  1566,    12,  5093,    10,    86,    48,   286,    92,    27,
           141,    82, 11457,     7,  1710,     6,    84,    27,  3218,   120,
          6002,    15,    26,    30,    21,    82,  2265,  1078,    13, 31078,
             6,    11,    84,    27,   470,  4567,    12,  8996,   182,  4321

In [20]:
trainer.train()

c:\Users\tyier\miniconda3\envs\qlora310\lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,2.920400
40,2.986700
60,3.038800
80,2.958800
100,2.920600
120,2.906000
140,2.978600
160,2.908900
180,2.895000
200,2.983200


c:\Users\tyier\miniconda3\envs\qlora310\lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\tyier\miniconda3\envs\qlora310\lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\tyier\min

TrainOutput(global_step=375, training_loss=2.9375044759114584, metrics={'train_runtime': 334.8489, 'train_samples_per_second': 17.919, 'train_steps_per_second': 1.12, 'total_flos': 143689732546560.0, 'train_loss': 2.9375044759114584, 'epoch': 3.0})

In [21]:
model.eval()

finetuned_predictions = []

for src in sources:
    pred = generate_translation(
        model,
        tokenizer,
        src,
    )
    finetuned_predictions.append(pred)

print("FINETUNED EXAMPLE:")
print(finetuned_predictions[0])

FINETUNED EXAMPLE:
Kitty, en el contrario, fue más activo que el habitado y incluso más animated.


In [22]:
finetuned_bleu = bleu.compute(
    predictions=finetuned_predictions,
    references=[[ref] for ref in references],
)

print("Fine-tuned BLEU:")
print(finetuned_bleu)

Fine-tuned BLEU:
{'score': 6.744219700547908, 'counts': [141, 41, 15, 6], 'totals': [366, 346, 326, 306], 'precisions': [38.52459016393443, 11.84971098265896, 4.601226993865031, 1.9607843137254901], 'bp': 0.841868756910948, 'sys_len': 366, 'ref_len': 429}


In [23]:
for i in range(5):
    print("=" * 80)
    print("SOURCE:    ", sources[i])
    print("REFERENCE: ", references[i])
    print("BASELINE:  ", baseline_predictions[i])
    print("QLORA FT:  ", finetuned_predictions[i])

SOURCE:     Kitty, on the contrary, was more active than usual and even more animated.
REFERENCE:  Kitty, al contrario, estaba más activa a incluso más animada que nunca.
BASELINE:   Kitty, en el contrario, fue más activo que el habitado y incluso más animated.
QLORA FT:   Kitty, en el contrario, fue más activo que el habitado y incluso más animated.
SOURCE:     All necessary preparations shall be made for your return.
REFERENCE:  Se darán las órdenes necesarias para su regreso.
BASELINE:   Todos los preocupaciones necesarios se han realizado para vuestro regreso.
QLORA FT:   Todas las preparaciones necesarias se han realizado para vuestra regresión.
SOURCE:     Maintenant partons, allons, allons vers Sion.
REFERENCE:  ¡Demasiado larga ha sido ya la pausa! ¡Adelante!
BASELINE:   Y ahora partemos, aguas, aguas versión.
QLORA FT:   Y ahora partemos, allons, allons versión de Sión.
SOURCE:     Thursday, Aug.
REFERENCE:  Jueves 20 de agosto.
BASELINE:   Seora, Aug.
QLORA FT:   Seora, Aug.


In [24]:
model.save_pretrained("./flan_t5_qlora_adapter")
tokenizer.save_pretrained("./flan_t5_qlora_adapter")

print("QLoRA adapter and tokenizer saved.")

QLoRA adapter and tokenizer saved.


In [25]:
results = {
    "model_id": model_id,
    "method": "QLoRA",
    "sources": sources,
    "references": references,
    "baseline_predictions": baseline_predictions,
    "finetuned_predictions": finetuned_predictions,
    "baseline_bleu": baseline_bleu,
    "finetuned_bleu": finetuned_bleu,
}

with open("flan_t5_qlora_translation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Results saved to flan_t5_qlora_translation_results.json")

Results saved to flan_t5_qlora_translation_results.json
